In [ ]:
import sqlite3
import pandas as pd
import networkx as nx
from pyvis.network import Network

# ==========================================
# 1. DATA EXTRACTION & PREPROCESSING
# ==========================================
def load_and_clean_data(db_path):
    conn = sqlite3.connect(db_path)
    query = """
    SELECT 
        COALESCE(ea1.canonical_name, t.actor) AS source,
        COALESCE(ea2.canonical_name, t.target) AS target
    FROM rdf_triples t
    LEFT JOIN entity_aliases ea1 ON t.actor = ea1.original_name
    LEFT JOIN entity_aliases ea2 ON t.target = ea2.original_name
    """
    df_edges = pd.read_sql_query(query, conn)
    conn.close()

    df_edges = df_edges.dropna(subset=['source', 'target'])

    # Rimozione artefatti di estrazione NLP
    extraction_artifacts = [
        'communication', 'association', 'lawsuit', 'document', 
        'email', 'letter', 'meeting', 'payment', 'none', 'unknown'
    ]
    
    df_clean = df_edges[
        (~df_edges['source'].str.lower().isin(extraction_artifacts)) & 
        (~df_edges['target'].str.lower().isin(extraction_artifacts)) &
        (df_edges['source'] != df_edges['target']) 
    ].copy()

    # Calcolo dei pesi degli archi (frequenza interazioni)
    edge_weights = df_clean.groupby(['source', 'target']).size().reset_index(name='weight')
    return edge_weights

# ==========================================
# 2. GRAPH CONSTRUCTION & METRICS (RQ2)
# ==========================================
def compute_network_metrics(edge_weights):
    G = nx.from_pandas_edgelist(edge_weights, 'source', 'target', ['weight'])
    
    degree_dict = nx.degree_centrality(G)
    
    # Campionamento k=100 per ottimizzazione computazionale su reti ampie
    betweenness_dict = nx.betweenness_centrality(G, k=100, weight='weight', seed=42) 

    nx.set_node_attributes(G, degree_dict, 'degree')
    nx.set_node_attributes(G, betweenness_dict, 'betweenness')

    df_metrics = pd.DataFrame({
        'Node': list(G.nodes()),
        'Degree': list(degree_dict.values()),
        'Betweenness': list(betweenness_dict.values())
    }).sort_values(by='Betweenness', ascending=False)
    
    return G, df_metrics

# ==========================================
# 3. ADVANCED VISUALIZATION
# ==========================================
def generate_interactive_network(G, output_filename='network_topology.html'):
    # Filtraggio dei nodi periferici per garantire leggibilità
    core_nodes = [node for node, degree in dict(G.degree()).items() if degree > 5]
    G_core = G.subgraph(core_nodes)

    net = Network(height='800px', width='100%', bgcolor='#1a1a1a', font_color='#ffffff', notebook=True)
    net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=100, spring_strength=0.08)

    betweenness_dict = nx.get_node_attributes(G_core, 'betweenness')

    for node in G_core.nodes():
        node_str = str(node).lower()
        size = (betweenness_dict.get(node, 0) * 1000) + 10 
        
        # Classificazione cromatica: evidenziazione entità anonime (Broker investigation)
        color = '#cc0000' if 'unknown' in node_str else '#2b7ce9'
            
        net.add_node(
            node, 
            label=str(node), 
            title=f"Betweenness: {betweenness_dict.get(node, 0):.4f}", 
            size=size, 
            color=color
        )

    for source, target, data in G_core.edges(data=True):
        net.add_edge(source, target, value=data['weight'], color='rgba(150,150,150,0.3)')

    net.show(output_filename)

# ==========================================
# ESECUZIONE PIPELINE
# ==========================================
if __name__ == "__main__":
    db_path = '../data/document_analysis.db'
    
    edge_weights = load_and_clean_data(db_path)
    G, df_metrics = compute_network_metrics(edge_weights)
    
    print("Top 10 Nodes by Betweenness Centrality:")
    print(df_metrics.head(10).to_string(index=False))
    
    generate_interactive_network(G, 'network_topology_rq2.html')